# Phase 2 — Simulated Annealing (SA)
### 3D Container Loading Optimizer

---
## What is Simulated Annealing?

Imagine you're trying to find the lowest point in a hilly landscape — blindfolded.
A simple strategy: always step downhill. But this gets you **stuck in small valleys** (local optima).

Simulated Annealing's strategy: **sometimes take a step uphill on purpose**, especially early on.
As time goes on, you take fewer and fewer uphill steps until you only go downhill.
This way you **escape local optima** and find a much better valley (global optimum).

The name comes from **metallurgy**: when you heat metal and let it cool slowly,
atoms settle into a strong, low-energy structure. Fast cooling = weak metal. Slow cooling = strong metal.
We do the same with our solution.

---
## How it works in our project:

1. Start with a random order (sequence) of boxes
2. Pack them into the container using a heuristic → measure how full it is (**fitness**)
3. **Perturb**: randomly swap two boxes in the sequence
4. Pack again → measure new fitness
5. If new is better → accept it
6. If new is worse → **maybe accept it anyway** (based on temperature)
7. Slowly lower the temperature → repeat thousands of times
8. Return the best solution found

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import random
import math
import time
from dataclasses import dataclass, field
from typing import List, Tuple, Optional
from copy import deepcopy

random.seed(42)
np.random.seed(42)

print('All imports OK ✅')

## 2. Load Data

We load the full 120-box dataset that was prepared in Phase 1.

In [ ]:
df = pd.read_csv('boxes_120(generated+local).csv')

# Make sure fragile column is boolean
df['fragile'] = df['fragile'].astype(str).str.strip().str.lower().map(
    {'true': True, 'false': False, '1': True, '0': False}
).fillna(False)

# Compute volume
df['volume_cm3'] = df['length'] * df['width'] * df['height']

print(f'Loaded {len(df)} boxes')
print(f'Fragile: {df["fragile"].sum()} | Non-fragile: {(~df["fragile"]).sum()}')
print(f'Total box volume: {df["volume_cm3"].sum():,.0f} cm³')
df.head()

## 3. Data Structures

We define the same `Box` and `Container` classes from Phase 1.
These are the building blocks used by all three algorithms.

In [ ]:
@dataclass
class Box:
    """
    Represents a single box.
    After packing, x/y/z store its position in the container.
    placed_orientation stores which L/W/H rotation was used.
    """
    id: int
    length: float
    width: float
    height: float
    weight_kg: float
    fragile: bool = False
    # Position (set during packing)
    x: float = 0.0
    y: float = 0.0
    z: float = 0.0
    placed: bool = False
    placed_orientation: Tuple = None

    @property
    def volume(self) -> float:
        return self.length * self.width * self.height

    def get_orientations(self) -> List[Tuple]:
        """
        Returns valid rotations for this box.
        Fragile boxes: only rotate horizontally (height stays vertical).
        Normal boxes: all 6 rotations allowed.
        """
        l, w, h = self.length, self.width, self.height
        if self.fragile:
            return [(l, w, h), (w, l, h)]
        else:
            return [
                (l, w, h), (l, h, w),
                (w, l, h), (w, h, l),
                (h, l, w), (h, w, l)
            ]

    def __repr__(self):
        return f'Box(id={self.id}, {self.length}x{self.width}x{self.height}, {self.weight_kg}kg, fragile={self.fragile})'


@dataclass
class Container:
    """
    ISO 20ft standard shipping container.
    Internal dimensions in cm.
    """
    length: float = 589.0
    width: float  = 235.0
    height: float = 239.0

    @property
    def volume(self) -> float:
        return self.length * self.width * self.height

    def __repr__(self):
        return f'Container({self.length}x{self.width}x{self.height}cm, {self.volume/1e6:.3f}m³)'


# Build Box objects from dataframe
def build_boxes(dataframe: pd.DataFrame) -> List[Box]:
    return [
        Box(
            id=int(row['id']),
            length=float(row['length']),
            width=float(row['width']),
            height=float(row['height']),
            weight_kg=float(row['weight_kg']),
            fragile=bool(row['fragile'])
        )
        for _, row in dataframe.iterrows()
    ]


CONTAINER = Container()
ALL_BOXES = build_boxes(df)

print(CONTAINER)
print(f'Built {len(ALL_BOXES)} Box objects')

## 4. The Packing Heuristic (Decoder)

### What is a Decoder?
The SA algorithm works with a **sequence** (ordering) of boxes — not positions.
The decoder takes that sequence and figures out WHERE to physically place each box.

We use the **Deepest Bottom-Left Fill (DBLF)** strategy:
- Maintain a list of **empty spaces** (called Extreme Points)
- For each box in the sequence, try every empty space and every valid rotation
- Place the box in the space that wastes the least room
- Update the list of empty spaces

### Gravity Constraint
A box can only be placed at height z=0 (the floor) OR on top of another box.
No floating boxes allowed!

In [ ]:
def check_overlap(box1_pos, box1_dim, box2_pos, box2_dim) -> bool:
    """
    Returns True if two boxes overlap.
    Each box is defined by its corner position (x,y,z) and dimensions (l,w,h).

    Two boxes do NOT overlap if one is completely to the left/right/front/back/above/below the other.
    We check all 6 cases — if ANY of them is true, there's no overlap.
    """
    x1, y1, z1 = box1_pos
    l1, w1, h1 = box1_dim
    x2, y2, z2 = box2_pos
    l2, w2, h2 = box2_dim

    # No overlap if separated along any axis
    if (x1 + l1 <= x2 or x2 + l2 <= x1 or
        y1 + w1 <= y2 or y2 + w2 <= y1 or
        z1 + h1 <= z2 or z2 + h2 <= z1):
        return False
    return True  # They overlap


def is_supported(x, y, z, l, w, placed_boxes) -> bool:
    """
    Gravity check: a box at height z must be either:
    - On the floor (z == 0), OR
    - Fully supported by boxes below it

    We check that the footprint of the new box is supported.
    """
    if z == 0:
        return True  # Floor supports everything

    # Check if there's a box directly below that covers enough area
    for pb in placed_boxes:
        px, py, pz = pb['pos']
        pl, pw, ph = pb['dim']
        # Does the box below reach exactly to z?
        if abs((pz + ph) - z) < 0.01:
            # Does it overlap the footprint?
            overlap_x = min(x + l, px + pl) - max(x, px)
            overlap_y = min(y + w, py + pw) - max(y, py)
            if overlap_x > 0 and overlap_y > 0:
                return True
    return False


def pack_sequence(sequence: List[Box], container: Container) -> Tuple[List[dict], float]:
    """
    THE DECODER: takes an ordered list of boxes and packs them into the container.

    Strategy: Bottom-Left-Fill
    - Try to place each box as far back-left-bottom as possible
    - Try all valid orientations
    - Respect: container boundaries, no overlap, gravity

    Returns:
        placed_boxes: list of dicts with position and dimension info
        utilization: percentage of container volume used (0-100)
    """
    placed_boxes = []
    # Candidate positions to try — start with the origin (0,0,0)
    candidate_points = [(0.0, 0.0, 0.0)]
    total_volume_packed = 0.0

    for box in sequence:
        best_position = None
        best_orientation = None
        best_score = float('inf')  # Lower = better (closer to origin)

        for orientation in box.get_orientations():
            bl, bw, bh = orientation

            for (cx, cy, cz) in candidate_points:
                # Check container boundaries
                if (cx + bl > container.length or
                    cy + bw > container.width  or
                    cz + bh > container.height):
                    continue

                # Check gravity
                if not is_supported(cx, cy, cz, bl, bw, placed_boxes):
                    continue

                # Check no overlap with already placed boxes
                overlaps = False
                for pb in placed_boxes:
                    if check_overlap((cx, cy, cz), (bl, bw, bh),
                                     pb['pos'], pb['dim']):
                        overlaps = True
                        break
                if overlaps:
                    continue

                # Score: prefer positions close to origin (bottom-left-front)
                # Heavy boxes get a bonus for being low (stability heuristic)
                stability_penalty = cz * box.weight_kg  # heavier + higher = worse
                score = cx + cy + cz + stability_penalty

                if score < best_score:
                    best_score = score
                    best_position = (cx, cy, cz)
                    best_orientation = orientation

        if best_position is not None:
            bx, by, bz = best_position
            bl, bw, bh = best_orientation

            placed_boxes.append({
                'id': box.id,
                'pos': best_position,
                'dim': best_orientation,
                'weight': box.weight_kg,
                'fragile': box.fragile
            })
            total_volume_packed += bl * bw * bh

            # Add new candidate points: the 3 corners created by placing this box
            candidate_points.append((bx + bl, by,      bz))
            candidate_points.append((bx,      by + bw, bz))
            candidate_points.append((bx,      by,      bz + bh))

    utilization = (total_volume_packed / container.volume) * 100.0
    return placed_boxes, utilization


# Quick test
test_seq = ALL_BOXES[:10]
placed, util = pack_sequence(test_seq, CONTAINER)
print(f'Test packing (10 boxes): placed {len(placed)}, utilization = {util:.2f}%')
print('Packing decoder works ✅')

## 5. Simulated Annealing Algorithm

### The key parameters:

| Parameter | What it means | Analogy |
|-----------|--------------|--------|
| `T_start` | Starting temperature | How hot the metal starts |
| `T_end` | Stopping temperature | How cold we let it get |
| `cooling_rate` | How fast we cool | Speed of cooling |
| `iterations` | Steps per temperature level | How much we stir at each temp |

### The acceptance formula:
When a new solution is **worse**, we accept it with probability:

$$P = e^{\frac{\Delta f}{T}}$$

Where:
- `Δf` = how much worse the new solution is
- `T` = current temperature
- When T is high → P is high → we accept bad moves often (exploring)
- When T is low → P is tiny → we almost never accept bad moves (exploiting)

In [ ]:
def simulated_annealing(
    boxes: List[Box],
    container: Container,
    T_start: float = 1000.0,
    T_end: float = 0.1,
    cooling_rate: float = 0.995,
    iterations_per_temp: int = 50
) -> dict:
    """
    Simulated Annealing for 3D Bin Packing.

    The 'solution' here is a SEQUENCE (ordering) of boxes.
    We perturb the sequence by swapping two random boxes,
    then decode it into an actual packing to get the fitness.

    Parameters:
        boxes            : list of all Box objects to pack
        container        : the Container
        T_start          : starting temperature
        T_end            : stopping temperature
        cooling_rate     : multiply temperature by this each step (0 < rate < 1)
        iterations_per_temp : how many swaps to try at each temperature

    Returns a dict with:
        best_sequence    : the best order of boxes found
        best_placed      : the actual packing result
        best_utilization : best volume utilization %
        history          : list of utilization values over time (for plotting)
        temperature_history : temperature at each step
        runtime          : total seconds taken
    """

    start_time = time.time()

    # ── Step 1: Create initial solution ──
    # Sort by volume descending (heaviest/largest first) as a smart starting point
    current_sequence = sorted(boxes, key=lambda b: b.volume, reverse=True)

    # Pack the initial sequence and measure its quality
    current_placed, current_util = pack_sequence(current_sequence, container)

    # Keep track of the best we've ever seen
    best_sequence    = current_sequence[:]
    best_placed      = current_placed
    best_utilization = current_util

    # ── Step 2: Track history for plotting ──
    history              = [current_util]
    temperature_history  = [T_start]
    best_history         = [best_utilization]

    T = T_start  # Current temperature
    total_iterations = 0
    accepted_worse   = 0  # Count how many bad moves we accepted

    # ── Step 3: Main SA loop ──
    while T > T_end:
        for _ in range(iterations_per_temp):
            total_iterations += 1

            # ── Perturbation: swap two random boxes ──
            # Copy current sequence so we don't destroy it
            new_sequence = current_sequence[:]
            i, j = random.sample(range(len(new_sequence)), 2)
            new_sequence[i], new_sequence[j] = new_sequence[j], new_sequence[i]

            # ── Decode: pack the new sequence ──
            new_placed, new_util = pack_sequence(new_sequence, container)

            # ── Decision: accept or reject? ──
            delta = new_util - current_util  # Positive = improvement

            if delta > 0:
                # New solution is BETTER → always accept
                current_sequence = new_sequence
                current_util     = new_util
                current_placed   = new_placed

            else:
                # New solution is WORSE → accept with probability e^(delta/T)
                # delta is negative here, so as T decreases, probability drops
                probability = math.exp(delta / T)
                if random.random() < probability:
                    current_sequence = new_sequence
                    current_util     = new_util
                    current_placed   = new_placed
                    accepted_worse  += 1

            # Update best ever seen
            if current_util > best_utilization:
                best_utilization = current_util
                best_sequence    = current_sequence[:]
                best_placed      = current_placed

            history.append(current_util)
            best_history.append(best_utilization)

        # ── Cooling ──
        T *= cooling_rate
        temperature_history.append(T)

    runtime = time.time() - start_time

    print(f'SA finished in {runtime:.1f}s')
    print(f'Total iterations    : {total_iterations:,}')
    print(f'Accepted worse moves: {accepted_worse:,}')
    print(f'Best utilization    : {best_utilization:.2f}%')
    print(f'Boxes placed        : {len(best_placed)} / {len(boxes)}')

    return {
        'best_sequence'     : best_sequence,
        'best_placed'       : best_placed,
        'best_utilization'  : best_utilization,
        'history'           : history,
        'best_history'      : best_history,
        'temperature_history': temperature_history,
        'runtime'           : runtime,
        'total_iterations'  : total_iterations
    }

## 6. Run the Algorithm

Now we actually run SA on our 120 boxes.
This may take 1–3 minutes depending on your computer.

In [ ]:
print('Starting Simulated Annealing...')
print(f'Container: {CONTAINER}')
print(f'Boxes to pack: {len(ALL_BOXES)}')
print('-' * 40)

sa_result = simulated_annealing(
    boxes               = ALL_BOXES,
    container           = CONTAINER,
    T_start             = 1000.0,
    T_end               = 0.1,
    cooling_rate        = 0.995,
    iterations_per_temp = 50
)

print('-' * 40)
print(f'✅ Done! Best utilization: {sa_result["best_utilization"]:.2f}%')

## 7. Visualize SA Performance

We plot:
1. How utilization evolved over all iterations
2. How temperature decreased over time
3. The best solution found at each step

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Simulated Annealing — Performance', fontsize=14, fontweight='bold')

history      = sa_result['history']
best_history = sa_result['best_history']
temp_history = sa_result['temperature_history']

# Plot 1: Utilization over iterations
axes[0].plot(history,      color='steelblue', alpha=0.5, linewidth=0.8, label='Current')
axes[0].plot(best_history, color='red',       linewidth=1.5,            label='Best so far')
axes[0].axhline(y=sa_result['best_utilization'], color='green',
                linestyle='--', linewidth=1.2, label=f'Best = {sa_result["best_utilization"]:.1f}%')
axes[0].set_title('Utilization Over Iterations')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Volume Utilization (%)')
axes[0].legend()

# Plot 2: Temperature schedule
axes[1].plot(temp_history, color='orange', linewidth=1.5)
axes[1].set_title('Temperature Cooling Schedule')
axes[1].set_xlabel('Temperature Step')
axes[1].set_ylabel('Temperature')
axes[1].set_yscale('log')  # Log scale shows the cooling curve better

# Plot 3: Summary bar chart
labels = ['Boxes Available', 'Boxes Placed']
values = [len(ALL_BOXES), len(sa_result['best_placed'])]
colors = ['#aaaaaa', '#4caf50']
bars = axes[2].bar(labels, values, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 str(val), ha='center', va='bottom', fontsize=12, fontweight='bold')
axes[2].set_title('Boxes Packed')
axes[2].set_ylabel('Count')
axes[2].set_ylim(0, len(ALL_BOXES) * 1.2)

plt.tight_layout()
plt.savefig('sa_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Performance plots saved ✅')

## 8. 3D Visualization of the Packing Result

We draw each placed box as a colored 3D rectangle inside the container.
- **Red boxes** = fragile
- **Blue boxes** = normal
- **Gray wireframe** = container boundary

You can rotate the 3D plot by clicking and dragging in the notebook.

In [ ]:
def draw_box_3d(ax, pos, dim, color, alpha=0.4, edgecolor='black'):
    """
    Draws a single 3D box on a matplotlib 3D axis.
    pos = (x, y, z) corner position
    dim = (l, w, h) dimensions
    """
    x, y, z = pos
    l, w, h = dim

    # Define the 8 corners of the box
    xx = [x, x+l, x+l, x, x, x+l, x+l, x]
    yy = [y, y,   y+w, y+w, y, y,   y+w, y+w]
    zz = [z, z,   z,   z, z+h, z+h, z+h, z+h]

    # Define the 6 faces using corner indices
    faces = [
        [0,1,2,3],  # bottom
        [4,5,6,7],  # top
        [0,1,5,4],  # front
        [2,3,7,6],  # back
        [0,3,7,4],  # left
        [1,2,6,5],  # right
    ]

    poly = Poly3DCollection(
        [[[xx[i], yy[i], zz[i]] for i in face] for face in faces],
        alpha=alpha, facecolor=color, edgecolor=edgecolor, linewidth=0.3
    )
    ax.add_collection3d(poly)


def visualize_packing_3d(placed_boxes, container, title='SA Packing Result'):
    """
    Full 3D visualization of all placed boxes inside the container.
    """
    fig = plt.figure(figsize=(14, 9))
    ax  = fig.add_subplot(111, projection='3d')

    # Draw container wireframe
    cl, cw, ch = container.length, container.width, container.height
    container_corners = [
        [0,0,0], [cl,0,0], [cl,cw,0], [0,cw,0],
        [0,0,ch],[cl,0,ch],[cl,cw,ch],[0,cw,ch]
    ]
    edges = [
        (0,1),(1,2),(2,3),(3,0),  # bottom
        (4,5),(5,6),(6,7),(7,4),  # top
        (0,4),(1,5),(2,6),(3,7)   # verticals
    ]
    for e in edges:
        p1, p2 = container_corners[e[0]], container_corners[e[1]]
        ax.plot([p1[0],p2[0]], [p1[1],p2[1]], [p1[2],p2[2]],
                'k--', alpha=0.3, linewidth=0.8)

    # Color palette — cycle through colors for variety
    normal_colors  = ['#4fc3f7','#81c784','#ffb74d','#ce93d8','#80cbc4','#fff176']
    fragile_colors = ['#ef5350','#ff7043','#ec407a']

    n_idx = 0
    f_idx = 0

    for pb in placed_boxes:
        if pb['fragile']:
            color = fragile_colors[f_idx % len(fragile_colors)]
            f_idx += 1
        else:
            color = normal_colors[n_idx % len(normal_colors)]
            n_idx += 1
        draw_box_3d(ax, pb['pos'], pb['dim'], color=color, alpha=0.6)

    # Axis labels and limits
    ax.set_xlim(0, cl)
    ax.set_ylim(0, cw)
    ax.set_zlim(0, ch)
    ax.set_xlabel('Length (cm)')
    ax.set_ylabel('Width (cm)')
    ax.set_zlabel('Height (cm)')

    # Legend
    normal_patch  = mpatches.Patch(color='#4fc3f7', label='Normal box')
    fragile_patch = mpatches.Patch(color='#ef5350', label='Fragile box')
    ax.legend(handles=[normal_patch, fragile_patch], loc='upper left')

    util = sum(pb['dim'][0]*pb['dim'][1]*pb['dim'][2] for pb in placed_boxes)
    util_pct = util / container.volume * 100
    ax.set_title(f'{title}\n{len(placed_boxes)} boxes placed | {util_pct:.1f}% utilization',
                 fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.savefig('sa_3d_packing.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('3D visualization saved ✅')


visualize_packing_3d(sa_result['best_placed'], CONTAINER)

## 9. Results Summary

In [ ]:
placed = sa_result['best_placed']

total_vol_packed = sum(p['dim'][0]*p['dim'][1]*p['dim'][2] for p in placed)
fragile_placed   = sum(1 for p in placed if p['fragile'])
heavy_boxes      = [p for p in placed if p['weight'] > 10]

# Check stability: are heavy boxes mostly at the bottom?
if heavy_boxes:
    avg_height_heavy = np.mean([p['pos'][2] for p in heavy_boxes])
    avg_height_all   = np.mean([p['pos'][2] for p in placed])
    stability_ok     = avg_height_heavy <= avg_height_all
else:
    stability_ok = True

print('=' * 50)
print('        SIMULATED ANNEALING — RESULTS')
print('=' * 50)
print(f'  Container volume     : {CONTAINER.volume/1e6:.3f} m³')
print(f'  Volume packed        : {total_vol_packed/1e6:.3f} m³')
print(f'  Utilization          : {sa_result["best_utilization"]:.2f}%')
print(f'  Boxes placed         : {len(placed)} / {len(ALL_BOXES)}')
print(f'  Fragile boxes placed : {fragile_placed}')
print(f'  Heavy boxes at bottom: {"✅ Yes" if stability_ok else "⚠️ Check"}')
print(f'  Runtime              : {sa_result["runtime"]:.1f}s')
print(f'  Total iterations     : {sa_result["total_iterations"]:,}')
print('=' * 50)

target = 75.0
if sa_result['best_utilization'] >= target:
    print(f'\n✅ SUCCESS: Utilization ≥ {target}% target!')
else:
    print(f'\n⚠️  Below {target}% target. Consider tuning parameters.')

## 10. Parameter Tuning Tips

If your utilization is below 75%, try adjusting these parameters in the `simulated_annealing()` call:

| Parameter | Default | Try this |
|-----------|---------|----------|
| `T_start` | 1000 | Increase to 5000 |
| `T_end` | 0.1 | Decrease to 0.01 |
| `cooling_rate` | 0.995 | Increase to 0.999 (slower cooling) |
| `iterations_per_temp` | 50 | Increase to 100 |

**Slower cooling = more exploration = better results, but takes longer.**

In [ ]:
# Export best packing plan to CSV for use in the desktop app
packing_plan = pd.DataFrame([
    {
        'box_id'   : p['id'],
        'x'        : p['pos'][0],
        'y'        : p['pos'][1],
        'z'        : p['pos'][2],
        'length'   : p['dim'][0],
        'width'    : p['dim'][1],
        'height'   : p['dim'][2],
        'weight_kg': p['weight'],
        'fragile'  : p['fragile']
    }
    for p in placed
])

packing_plan.to_csv('sa_packing_plan.csv', index=False)
print(f'Packing plan saved to sa_packing_plan.csv ✅')
print(f'Shape: {packing_plan.shape}')
packing_plan.head(10)